# Demo 2
## Controlled Pendulum
### PyFMI Co-Simulation

**Modelica Models**

In [8]:
from path import Path

root = Path(r'/home/flo/repos/SystemSimulation/demos/ControlledPendulum/')
pkg_dir = Path(root / "ControlledPendulum")
pkg_str = str(root / "ControlledPendulum" / "package.mo")
pkg_name = "ControlledPendulum"

model_names = Path(pkg_dir).files("*.mo")
model_names = [m.stem for m in model_names if m.stem != "package"]

# Sort models to have a consistent order
model_names.sort()

for model_name in model_names:
    print(f"Found model: {model_name}")

Found model: AngleEncoder
Found model: Demo_Driven
Found model: Demo_DrivenWithWall
Found model: Demo_UndrivenWallDiscrete
Found model: Demo_UndrivenWithWall
Found model: Drive
Found model: ImpactWall
Found model: PID_Continuous
Found model: PID_Sampled
Found model: Pendulum
Found model: Reference


**Helper Functions for FMU Creation**

In [9]:
from OMPython import ModelicaSystem
import shutil

def create_modelica_system(model_name):
    model = ModelicaSystem(pkg_str, model_name, verbose=True)
    return model

def create_fmu(model: ModelicaSystem, fmuType="cs"):
    model.buildModel()
    fmu_path = model.convertMo2Fmu(fmuType=fmuType)
    return fmu_path

def move_file(src_path, dst_path):
    shutil.move(src_path, dst_path)

**Create FMUs for the Models**

In [10]:
# Check if fmus exist, if not create them
fmu_dir = root / "FMUs"
if not fmu_dir.exists():
    fmu_dir.mkdir()
fmu_path_dict = {}

for model_name in model_names:
    fmu_path = fmu_dir / f"{model_name}.fmu"
    if not fmu_path.exists():
        print(f"Creating FMU for model: {model_name}")
        model = create_modelica_system(pkg_name + "." + model_name)
        temp_fmu_path = create_fmu(model)
        move_file(temp_fmu_path, fmu_path)
    else:
        print(f"FMU already exists for model: {model_name}")
    fmu_path_dict[model_name] = str(fmu_path)

FMU already exists for model: AngleEncoder
FMU already exists for model: Demo_Driven
FMU already exists for model: Demo_DrivenWithWall
FMU already exists for model: Demo_UndrivenWallDiscrete
FMU already exists for model: Demo_UndrivenWithWall
FMU already exists for model: Drive
FMU already exists for model: ImpactWall
FMU already exists for model: PID_Continuous
FMU already exists for model: PID_Sampled
FMU already exists for model: Pendulum
FMU already exists for model: Reference


In [11]:
from pyfmi import load_fmu

ref_fmu = load_fmu(fmu_path_dict["Reference"])
sensor_ref_fmu = load_fmu(fmu_path_dict["AngleEncoder"])
sensor_state_fmu = load_fmu(fmu_path_dict["AngleEncoder"])
pid_fmu = load_fmu(fmu_path_dict["PID_Continuous"])
drive_fmu = load_fmu(fmu_path_dict["Drive"])
pendulum_fmu = load_fmu(fmu_path_dict["Pendulum"])

fmu_list = [ref_fmu, sensor_ref_fmu, sensor_state_fmu, pid_fmu, drive_fmu, pendulum_fmu]

t = 0.0
dt = 0.001
tf = 10.0
h = dt  # step size

for fmu in fmu_list:
    fmu.reset()
    fmu.setup_experiment(start_time=t)
    fmu.initialize()

**Co-Simulation Loop**

In [12]:
# Initialize logging arrays
ts = []
q_ref_log, q_state_log, omega_state_log = [], [], []
U_ref_log, U_state_log = [], []

while t < tf - 1e-9:
    # 1) Read the current outputs
    ref_fmu.do_step(current_t=t, step_size=h)
    q_ref = float(ref_fmu.get("q_ref")[0])

    # 2) Read plant state at current time
    q_state = float(pendulum_fmu.get("q_state")[0])
    omega_state = float(pendulum_fmu.get("omega_state")[0])

    # 3) Sensors: set inputs -> step -> read outputs
    sensor_ref_fmu.set("q", q_ref)
    sensor_state_fmu.set("q", q_state)
    sensor_ref_fmu.do_step(current_t=t, step_size=h)
    sensor_state_fmu.do_step(current_t=t, step_size=h)
    U_ref = float(sensor_ref_fmu.get("U_q")[0])
    U_state = float(sensor_state_fmu.get("U_q")[0])

    error = U_ref - U_state

    # 4) PID: useses sensor voltages as inputs
    pid_fmu.set("y", U_state)
    pid_fmu.set("ref", U_ref)
    pid_fmu.do_step(current_t=t, step_size=h)
    u_control = float(pid_fmu.get("u")[0])

    # 5) Drive
    drive_fmu.set("u_control", u_control)
    drive_fmu.set("omega", omega_state)
    drive_fmu.do_step(current_t=t, step_size=h)
    torque = float(drive_fmu.get("torque")[0])

    # 6) Pendulum
    pendulum_fmu.set("torque", torque)
    pendulum_fmu.do_step(current_t=t, step_size=h)

    # Logging
    ts.append(t)
    q_ref_log.append(q_ref)
    q_state_log.append(q_state)
    omega_state_log.append(omega_state)
    U_ref_log.append(U_ref)
    U_state_log.append(U_state)

    # Increase time
    t += h

In [14]:
# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts, y=q_ref_log, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=ts, y=q_state_log, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(title='Pendulum Angle Tracking',
                  xaxis_title='Time (s)',
                  yaxis_title='Angle (degrees)',
                  legend_title='Legend',
                  template='plotly_dark')
fig.show()